# ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করা (Azure OpenAI)

LLM কেবলমাত্র টেক্সট জেনারেশনের জন্য নয়। আপনি টেক্সট বর্ণনা থেকে ইমেজও তৈরি করতে পারেন। ইমেজ এক ধরনের মাধ্যম হিসেবে মেডটেক, স্থাপত্য, পর্যটন, গেম ডেভেলপমেন্ট, মার্কেটিং এবং আরো বিভিন্ন ক্ষেত্রে ব্যবহারযোগ্য। এই পাঠে আমরা আজকের **GPT ইমেজ** মডেলগুলোর সাথে Microsoft Foundry-তে কাজ করব।

## শেখার লক্ষ্যসমূহ

এই পাঠের শেষে আপনি সক্ষম হবেন:

- ইমেজ জেনারেশন কি এবং কোথায় এটি ব্যবহারযোগ্য তা ব্যাখ্যা করতে।
- `gpt-image` মডেল পরিবারের সম্পর্কে বুঝতে এবং এটি পুরানো DALL·E মডেল থেকে কীভাবে আলাদা তা জানতে।
- একটি ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করতে এবং ইমেজ এডিট করতে।

## ইমেজ জেনারেশন বলতে কী বোঝায়?

ইমেজ জেনারেশন মডেলগুলি একটি টেক্সট প্রম্পট থেকে ছবি তৈরি করে। আধুনিক মডেল যেমন `gpt-image` প্রশিক্ষণের সময় টেক্সট ও ইমেজের সম্পর্ক শিখে, তারপর পুনরাবৃত্তিমূলক ভাবে র্যান্ডম নয়েজকে এমন একটি ছবিতে পরিণত করে যা আপনার প্রম্পটের সাথে মেলে।

দুটি পরিচিত ইমেজ মডেল পরিবার হলো:

- **`gpt-image` (OpenAI)** — এই পাঠে ব্যবহৃত বর্তমান প্রজন্ম। এটি টেক্সট-টু-ইমেজ জেনারেশন এবং ইমেজ এডিটিং (মাস্ক সহ ইনপেইন্টিং) সমর্থন করে।
- **Midjourney** — একটি জনপ্রিয় থার্ড-পার্টি মডেল যার নিজস্ব সার্ভিস এবং Discord-ভিত্তিক ওয়ার্কফ্লো রয়েছে।

> পুরনো OpenAI ইমেজ মডেলগুলো — **DALL·E 2** এবং **DALL·E 3** — আর লিগেসি। DALL·E 3 নতুন ডিপ্লয়মেন্টে আর উপলব্ধ নয়, এবং `create_variation` মত ফিচার শুধু DALL·E 2 তে ছিল। নতুন অ্যাপ্লিকেশনের জন্য `gpt-image` মডেল ব্যবহার করুন।

Microsoft Foundry-তে, **`gpt-image-2`** সর্বশেষ ও সবচেয়ে সক্ষম ইমেজ মডেল এবং এটি সুপারিশকৃত ডিফল্ট। `gpt-image-1.5` এবং `gpt-image-1-mini` ও সাধারণত উপলব্ধ।

> **গুরুত্বপূর্ণ:** `gpt-image` মডেলগুলো তৈরি করা ছবিটি **base64** (`b64_json`) হিসেবে ফিরে দেয়, URL হিসেবে নয়। আপনার কোড base64 স্ট্রিংকে বাইটে ডিকোড করে সংরক্ষণ করে — ডাউনলোড করার জন্য কোনো ইমেজ URL নেই।


## আপনার প্রথম ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করা

তাহলে ইমেজ জেনারেশন অ্যাপ্লিকেশন তৈরি করতে কি লাগে? আপনার নিম্নলিখিত লাইব্রেরিগুলি দরকার:

- **python-dotenv**, এই লাইব্রেরিটি ব্যবহার করার ব্যাপারে আপনাকে বিশেষভাবে পরামর্শ দেওয়া হচ্ছে যাতে আপনার গোপনীয় তথ্য কোড থেকে দূরে *.env* ফাইলে রাখা যায়।
- **openai**, এই লাইব্রেরি আপনি OpenAI API এর সাথে যোগাযোগ করার জন্য ব্যবহার করবেন।
- **pillow**, পাইথনে ইমেজের সাথে কাজ করার জন্য।

যদি এখনও না করে থাকেন, তাহলে [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) পৃষ্ঠায় নির্দেশনাগুলি অনুসরণ করুন একটি Microsoft Foundry রিসোর্স এবং মডেল তৈরির জন্য। মডেল হিসাবে **gpt-image-2** নির্বাচন করুন (সর্বশেষ Azure OpenAI ইমেজ মডেল; DALL·E হলো পুরাতনী)।

১. একটি *.env* নামক ফাইল তৈরি করুন নিচের বিষয়বস্তু নিয়ে:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    আপনার রিসোর্সের জন্য Microsoft Foundry পোর্টালে "Deployments" সেকশনে এই তথ্য খুঁজে পান।



1. উপরের লাইব্রেরিগুলো *requirements.txt* নামে একটি ফাইলে নিচের মত করে সংগ্রহ করুন:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. পরবর্তীতে, একটি ভার্চুয়াল এনভায়রনমেন্ট তৈরি করুন এবং লাইব্রেরিগুলো ইনস্টল করুন:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> উইন্ডোজের জন্য, আপনার ভার্চুয়াল এনভায়রনমেন্ট তৈরি এবং চালু করতে নিম্নলিখিত কমান্ডগুলি ব্যবহার করুন:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* নামের ফাইলে নিম্নলিখিত কোডটি যোগ করুন:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenv আমদানি করুন
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) ক্লায়েন্ট কনফিগার করুন।
    # ইমেজ মডেলগুলির ক্ষেত্রে সাম্প্রতিক API সংস্করণ প্রয়োজন — আপনার মডেল যে সংস্করণটি চায় তা Foundry ডকুমেন্টেশনে পরীক্ষা করুন।
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # ইমেজ তৈরি করতে ইমেজ জেনারেশন API ব্যবহার করুন
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # এখানে আপনার প্রম্ট টেক্সট লিখুন
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # উদাহরণস্বরূপ "gpt-image-2"
        )
        # সংরক্ষিত ইমেজের ডিরেক্টরি নির্ধারণ করুন
        image_dir = os.path.join(os.curdir, 'images')

        # যদি ডিরেক্টরি না থাকে, তবে এটি তৈরি করুন
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # ইমেজ পাথ ইনিশিয়ালাইজ করুন (মনে রাখবেন ফাইলের ধরন png হওয়া উচিত)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image মডেলগুলি ইমেজটি URL নয়, base64 (b64_json) হিসাবে ফিরিয়ে দেয়
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # ডিফল্ট ইমেজ ভিউয়ারে ইমেজ প্রদর্শন করুন
        image = Image.open(image_path)
        image.show()

    # ব্যতিক্রম ধরা হোক
    except BadRequestError as err:
        print(err)

    ```

আসুন এই কোডটি ব্যাখ্যা করি:

- প্রথমত, আমরা প্রয়োজনীয় লাইব্রেরি গুলো ইম্পোর্ট করি, যার মধ্যে থাকে OpenAI লাইব্রেরি, dotenv লাইব্রেরি, base64 মডিউল, এবং Pillow লাইব্রেরি।

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- এরপর, আমরা *.env* ফাইল থেকে পরিবেশ ভেরিয়েবল লোড করি।

    ```python
    # ডটএনভি ইমপোর্ট করুন
    dotenv.load_dotenv()
    ```

- তারপরে, আমরা Azure OpenAI (Microsoft Foundry) ক্লায়েন্ট কনফিগার করি।

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- এরপর, আমরা ছবি তৈরি করি:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # এখানে আপনার প্রম্পট পাঠ্য প্রবেশ করান
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` মডেলগুলো ছবি **base64** স্ট্রিং হিসেবে `data[0].b64_json` এ রিটার্ন করে। আমরা এটিকে বাইটে ডিকোড করি এবং একটি ফাইলে লিখি — এখানে ডাউনলোড করার জন্য URL থাকে না।

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- সবশেষে, আমরা ছবিটি খুলে স্ট্যান্ডার্ড ইমেজ ভিউয়ার ব্যবহার করে প্রদর্শন করি:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### ছবির উৎপাদন সম্পর্কিত আরও বিস্তারিত

আসুন `images.generate` এর প্যারামিটারগুলো দেখি:

- **prompt**, হচ্ছে টেক্সট প্রম্পট যেটি ছবিটি তৈরি করতে ব্যবহার করা হয়। এখানে এটি "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils"।
- **size**, হচ্ছে তৈরি হওয়া ছবির আকার (যেমন `1024x1024`, `1536x1024`, `1024x1536`, অথবা `"auto"`)।
- **n**, হচ্ছে তৈরি হওয়া ছবির সংখ্যা। এখানে আমরা একটি তৈরি করেছি।
- **model**, হচ্ছে আপনার ইমেজ মডেলের ডিপ্লয়মেন্ট নাম (যেমন `gpt-image-2`)।

> ইমেজ মডেলগুলো `temperature` প্যারামিটার নেয় না — এটি টেক্সট-জেনারেশনের নিয়ন্ত্রণ। বৈচিত্র্যের জন্য API আবার কল করুন; বৈচিত্র্য কমাতে, আপনার প্রম্পট আরও বিশেষভাবে করুন।

## ইমেজ জেনারেশনের অতিরিক্ত ক্ষমতা

আপনি দেখেছেন কীভাবে কয়েকটি পাইথন লাইনে ছবি তৈরি করা যায়। `gpt-image` মডেলগুলো একটি বিদ্যমান ছবিও **সম্পাদনা** করতে পারে। একটি ছবি, ঐচ্ছিক **মাস্ক** (যা পরিবর্তন করার এলাকা চিহ্নিত করে), এবং একটি প্রম্পট দিয়ে, আপনি ছবির একটি অংশ পরিবর্তন করতে পারেন — যেমন, আমাদের খরগোশের মাথায় টুপি যোগ করা।

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# সম্পাদনাগুলিও base64 হিসাবে ফেরত দেওয়া হয়
edited_image = base64.b64decode(response.data[0].b64_json)
```

বেস ছবি শুধুমাত্র খরগোশের; চূড়ান্ত ছবিতে খরগোশের মাথায় টুপি রয়েছে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
